# PKG edge-aggregate sizing probe: storage and build time, with vs. without debit card

**Target table (not built yet).**
`(txn_date, src_id, src_kind, dst_id, dst_kind, category) → amount, n_txn`
- ids are `mdm_id` for customers and `unq_cpty_acct_id` for counterparties
- 5C/6C duplicates are removed first, using the v2 rule

**The probe does five things:**
1. Measures your HDFS space: quota, used, remaining.
2. Measures the raw staging table per category, over full history and for the probe month.
3. Builds the probe month at **daily** grain, writes it to your HDFS space, and rolls it up to **weekly** and **monthly** grain.
4. Measures exact key counts and compressed bytes per category and grain.
5. Extrapolates to full history per category, then sums with and without `DEBIT_CARD`.

Storage adds up by category, so a single run answers both scenarios. The probe output is deleted at the end (`CLEANUP`).

In [ ]:
# ---- Config -------------------------------------------------------------------------------
TABLE         = "dsihd01p_dsi.neo4j_payments"
PROBE_MONTH   = "2025-01"
PROBE_DAYS    = None           # e.g. 3 → only days 1..3 (weekly/monthly figures then NOT representative)

HDFS_HOME     = "hdfs://nameservice1/user/pk36814/"
PROBE_DIR     = HDFS_HOME.rstrip("/") + "/pkg_tmp/agg_sizing_probe"
CLEANUP       = True           # delete probe output after measuring

CARD_PATTERN  = "DEBIT_CARD"   # categories containing this = the card group (8B, 8C)
KEEP_CAT, DROP_CAT = "5C.RTP_PRT_CPTY_PAYS", "6C.RTP_P2P_CPTY_PAYS"

CODEC             = "snappy"   # rerun with "zstd" to compare
TARGET_FILE_BYTES = 256 * 1024**2   # file sizing: ≈ one output file per 256 MB of raw input per category
COUNT_HISTORY     = True       # rows per month over full history (parquet footers only)

In [ ]:
# ---- NumPy compat shim (must run before any toPandas) ------------------------------------
import numpy as np, warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning, module=r"pyspark.*")

from pyspark.sql import SparkSession, Window, functions as F
from pyspark import StorageLevel
from IPython.display import display
from urllib.parse import unquote
from contextlib import contextmanager
import pandas as pd, time, calendar, datetime as dt

spark = SparkSession.builder.getOrCreate()
print("Spark", spark.version, "| pandas", pd.__version__, "| numpy", np.__version__)

HCONF = spark._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
fs_for = lambda p: JPath(p).getFileSystem(HCONF)
GB = 1e9

pd.set_option("display.max_rows", 200); pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)

def show(pdf, title, pct=(), gb=()):
    print(f"\n■ {title}   [{len(pdf):,} rows]")
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                fmt[c] = "{:.2%}"
        elif c in gb:                               fmt[c] = "{:,.1f}"
        elif pd.api.types.is_integer_dtype(pdf[c]): fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):   fmt[c] = "{:,.2f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:    sty = sty.hide(axis="index")
    except AttributeError: sty = sty.hide_index()
    display(sty)

TIMINGS = []
@contextmanager
def timed(stage):
    t0 = time.time()
    yield
    s = time.time() - t0
    TIMINGS.append({"stage": stage, "seconds": s})
    print(f"  ⏱ {stage}: {s/60:,.1f} min")

def nz(c):
    s = F.trim(F.col(c).cast("string"))
    return F.when(s == "", F.lit(None)).otherwise(s)

LO = f"{PROBE_MONTH}-01"
_y, _m = map(int, PROBE_MONTH.split("-"))
HI = f"{PROBE_MONTH}-{(PROBE_DAYS or calendar.monthrange(_y, _m)[1]):02d}"
is_card = lambda c: CARD_PATTERN in c
print(f"Probe window {LO} → {HI}")

## §0 Your HDFS space

HDFS quotas count **replicated** bytes. A 100 GB table at replication 3 consumes 300 GB of quota. Every "consumed" figure below includes replication.

In [ ]:
home = JPath(HDFS_HOME); fs_home = fs_for(HDFS_HOME)
REPL = int(fs_home.getDefaultReplication(home))

def quota_of(p, allow_summary=True):
    """(space_quota, space_consumed, ns_quota, ns_used). getQuotaUsage is O(1) on the NameNode;
    getContentSummary walks the subtree, so it is only allowed on the home dir, never on parents."""
    try:
        q = fs_home.getQuotaUsage(p)
        return q.getSpaceQuota(), q.getSpaceConsumed(), q.getQuota(), q.getFileAndDirectoryCount()
    except Exception:
        if not allow_summary: raise
        c = fs_home.getContentSummary(p)
        return c.getSpaceQuota(), c.getSpaceConsumed(), c.getQuota(), c.getFileCount() + c.getDirectoryCount()

cs = fs_home.getContentSummary(home)
sq, consumed, nsq, nsu = quota_of(home)

# A quota can sit on a parent (e.g. /user) instead of the home dir: walk up and report the first one found
quota_path, p = (str(home) if sq >= 0 else None), home
while quota_path is None and p.getParent() is not None:
    p = p.getParent()
    try:
        _sq, _cons, _, _ = quota_of(p, allow_summary=False)
        if _sq >= 0: quota_path, sq, consumed = str(p), _sq, _cons
    except Exception:
        break

stat = fs_home.getStatus(home)
REMAIN_RAW = (sq - consumed) if sq >= 0 else stat.getRemaining()
rows = [
    ("home logical size (what your files hold)",         f"{cs.getLength()/GB:,.1f} GB"),
    ("home consumed incl. replication",                  f"{cs.getSpaceConsumed()/GB:,.1f} GB"),
    ("replication factor",                               f"{REPL}"),
    ("space quota applies at",                           quota_path or "no quota on home or any parent"),
    ("space quota (replicated bytes)",                   f"{sq/GB:,.1f} GB" if sq >= 0 else "none"),
    ("quota consumed",                                   f"{consumed/GB:,.1f} GB ({consumed/sq:.1%})" if sq >= 0 else "—"),
    ("REMAINING (replicated)",                           f"{REMAIN_RAW/GB:,.1f} GB" + ("" if sq >= 0 else "  ← cluster free space, SHARED")),
    ("REMAINING as logical table size (÷ replication)",  f"{REMAIN_RAW/REPL/GB:,.1f} GB"),
    ("file/dir count quota",                             f"{nsu:,} used of {nsq:,}" if nsq >= 0 else f"{nsu:,} used, no quota"),
    ("cluster capacity / used / free",                   f"{stat.getCapacity()/GB/1000:,.1f} / {stat.getUsed()/GB/1000:,.1f} / {stat.getRemaining()/GB/1000:,.1f} TB"),
]
show(pd.DataFrame(rows, columns=["item", "value"]), f"HDFS space: {HDFS_HOME}")

## §1 Raw staging table: bytes per category, full history vs. probe month

In [ ]:
_f = (spark.table(TABLE).filter((F.col("category") == KEEP_CAT) & F.col("trans_dt").between(LO, HI)).inputFiles())
assert _f, f"no files for {KEEP_CAT} in {LO}..{HI}: check PROBE_MONTH"
ROOT = _f[0].split("/category=")[0]
fs_root = fs_for(ROOT)
print("table root:", ROOT)

raw = []
for s in fs_root.listStatus(JPath(ROOT)):
    name = s.getPath().getName()
    if not (s.isDirectory() and name.startswith("category=")): continue
    cat = unquote(name.split("=", 1)[1])
    full = fs_root.getContentSummary(s.getPath()).getLength()
    probe = 0
    for d in fs_root.listStatus(s.getPath()):
        v = unquote(d.getPath().getName().split("=", 1)[-1])
        if LO <= v[:10] <= HI:
            probe += fs_root.getContentSummary(d.getPath()).getLength()
    raw.append({"category": cat, "card": is_card(cat), "raw_full_bytes": full, "raw_probe_bytes": probe})
RAW = pd.DataFrame(raw).sort_values("category").reset_index(drop=True)

t = RAW.assign(raw_full_GB=RAW.raw_full_bytes / GB, raw_probe_GB=RAW.raw_probe_bytes / GB,
               share_full=RAW.raw_full_bytes / RAW.raw_full_bytes.sum())
show(t[["category", "card", "raw_full_GB", "share_full", "raw_probe_GB"]],
     f"Raw staging bytes per category (logical, excl. replication); probe = {LO}..{HI}",
     pct=("share_full",), gb=("raw_full_GB", "raw_probe_GB"))
for lbl, m in (("with card", RAW.card | ~RAW.card), ("without card", ~RAW.card)):
    print(f"  {lbl:<13} full history {RAW[m].raw_full_bytes.sum()/GB:,.0f} GB | probe {RAW[m].raw_probe_bytes.sum()/GB:,.1f} GB")

In [ ]:
# Rows per month over full history, card vs. rest (no data columns read → parquet footers)
if COUNT_HISTORY:
    with timed("history row count"):
        hist = (spark.table(TABLE)
                  .groupBy(F.substring("trans_dt", 1, 7).alias("month"),
                           F.col("category").contains(CARD_PATTERN).alias("card"))
                  .count().toPandas())
    H = hist.pivot(index="month", columns="card", values="count").fillna(0).astype("int64")
    H.columns = ["no_card" if not c else "card" for c in H.columns]
    H["total"] = H.sum(axis=1); H["card_share"] = H.get("card", 0) / H.total
    print(f"months: {len(H)} ({H.index.min()} → {H.index.max()}) | rows total {H.total.sum():,} | "
          f"without card {H.no_card.sum():,} ({H.no_card.sum()/H.total.sum():.1%})")
    show(H.tail(15).reset_index(), "Rows per month, last 15", pct=("card_share",))

## §2 Dedup 5C/6C for the probe window (v2 rule)

The key is `(date, receiving account, amount, upper-cased cpty_name)`, with one-to-one pairing and 5C kept.
- Tier 1 pairs rows on the same day.
- Tier 2 pairs a 6C row with a 5C row one day earlier.

5C/6C rows are read from **one day before** the window start, so tier-2 pairs that cross the month edge are found. This is the same function the full build will loop over.

In [ ]:
def dedup_drop_ids(lo, hi):
    """trans_ids of DROP_CAT rows in [lo, hi] that duplicate a KEEP_CAT row."""
    lo_ext = (dt.date.fromisoformat(lo) - dt.timedelta(days=1)).isoformat()
    r = (spark.table(TABLE)
           .filter(F.col("trans_dt").between(lo_ext, hi) & F.col("category").isin(KEEP_CAT, DROP_CAT))
           .select(nz("trans_id").alias("trans_id"), F.to_date("trans_dt").alias("dt"), nz("category").alias("category"),
                   nz("pnc_dep_acct_receives").alias("acct"), F.abs(F.col("trans_amt").cast("decimal(18,2)")).alias("amt"),
                   F.upper(F.regexp_replace(nz("cpty_name"), r"\s+", " ")).alias("nm"))
           .filter(F.col("acct").isNotNull() & F.col("nm").isNotNull())
           .withColumn("dt0", F.col("dt")))
    A, B = r.filter(F.col("category") == KEEP_CAT), r.filter(F.col("category") == DROP_CAT)
    GK = ["dt", "acct", "amt", "nm"]
    def ranked(df, s):
        w = Window.partitionBy(*GK).orderBy("trans_id")
        return df.withColumn("rn", F.row_number().over(w)).select(*GK, "rn", F.col("trans_id").alias(f"tid_{s}"),
                                                                  F.col("dt0").alias(f"dt0_{s}"))
    t1 = ranked(A, "a").join(ranked(B, "b"), GK + ["rn"]).withColumn("tier", F.lit(1))
    A_left = A.join(t1.select(F.col("tid_a").alias("trans_id")), "trans_id", "left_anti")
    B_left = B.join(t1.select(F.col("tid_b").alias("trans_id")), "trans_id", "left_anti")
    t2 = (ranked(A_left, "a").join(ranked(B_left.withColumn("dt", F.date_sub("dt", 1)), "b"), GK + ["rn"])
            .withColumn("tier", F.lit(2)))
    return (t1.unionByName(t2).filter(F.col("dt0_b") >= F.lit(lo).cast("date"))
              .select(F.col("tid_b").alias("trans_id"), "tier"))

with timed("dedup (5C/6C pairing)"):
    DROP_IDS = dedup_drop_ids(LO, HI).persist(StorageLevel.MEMORY_AND_DISK)
    dstat = DROP_IDS.groupBy("tier").count().orderBy("tier").toPandas()
N_DROP = int(dstat["count"].sum())
show(dstat, f"6C rows removed as duplicates in {LO}..{HI} (total {N_DROP:,})")

## §3 Build the probe month: daily, then weekly and monthly roll-ups

These are written to `PROBE_DIR`, partitioned by category and sorted by `(src_id, dst_id, txn_date)` inside each file. Sorting is what makes the id columns compress well.

In [ ]:
need_raw = 2.0 * RAW.raw_probe_bytes.sum() * REPL        # upper bound: daily ≤ raw; weekly + monthly < daily
print(f"space check: probe needs ≤ {need_raw/GB:,.0f} GB replicated | remaining {REMAIN_RAW/GB:,.0f} GB")
assert need_raw < REMAIN_RAW, "not enough space for the probe: set PROBE_DAYS, or point PROBE_DIR elsewhere"

def edges(df):
    return df.select(nz("trans_id").alias("trans_id"), F.to_date("trans_dt").alias("txn_date"),
                     nz("category").alias("category"),
                     nz("mdm_id_pays").alias("mp"), nz("mdm_id_receives").alias("mr"), nz("unq_cpty_acct_id").alias("cid"),
                     F.abs(F.col("trans_amt").cast("decimal(18,2)")).alias("amt"))

win = spark.table(TABLE).filter(F.col("trans_dt").between(LO, HI))
base = (edges(win.filter(F.col("category") != DROP_CAT))
          .unionByName(edges(win.filter(F.col("category") == DROP_CAT)).join(DROP_IDS.select("trans_id"), "trans_id", "left_anti")))

KEYS = ["src_id", "src_kind", "dst_id", "dst_kind", "category"]
daily = (base
         .withColumn("src_kind", F.when(F.col("mp").isNotNull(), "CUST").otherwise("CPTY"))
         .withColumn("src_id",   F.when(F.col("mp").isNotNull(), F.col("mp")).otherwise(F.col("cid")))
         .withColumn("dst_kind", F.when(F.col("mr").isNotNull(), "CUST").otherwise("CPTY"))
         .withColumn("dst_id",   F.when(F.col("mr").isNotNull(), F.col("mr")).otherwise(F.col("cid")))
         .groupBy("txn_date", *KEYS)
         .agg(F.sum("amt").cast("decimal(18,2)").alias("amount"), F.count("*").cast("int").alias("n_txn")))

# File sizing from raw bytes (known from §1), so no extra counting pass: n files per category ∝ raw size
NFILES = {r.category: max(1, round(r.raw_probe_bytes / TARGET_FILE_BYTES)) for r in RAW.itertuples()}
def write_agg(df, path, scale=1.0):
    nmap = F.create_map(*[x for c, n in NFILES.items() for x in (F.lit(c), F.lit(max(1, int(round(n * scale)))))])
    total = sum(max(1, int(round(n * scale))) for n in NFILES.values())
    (df.withColumn("_salt", F.pmod(F.hash("src_id"), F.coalesce(nmap[F.col("category")], F.lit(1))))
       .repartition(total, "category", "_salt").drop("_salt")
       .sortWithinPartitions("src_id", "dst_id", F.col(df.columns[0]))
       .write.mode("overwrite").option("compression", CODEC).partitionBy("category").parquet(path))

def bytes_by_category(path):
    fs, out = fs_for(path), {}
    for s in fs.listStatus(JPath(path)):
        n = s.getPath().getName()
        if s.isDirectory() and n.startswith("category="):
            c = fs.getContentSummary(s.getPath())
            out[unquote(n.split("=", 1)[1])] = (c.getLength(), c.getFileCount())
    return out

P = {g: f"{PROBE_DIR}/{g}" for g in ("daily", "weekly", "monthly")}
with timed("daily: scan + dedup join + aggregate + write"):
    write_agg(daily, P["daily"])

In [ ]:
d = spark.read.parquet(P["daily"])
AGG_COLS = ["amount", "n_txn"]
roll = lambda col, expr: (d.withColumn(col, expr).groupBy(col, *KEYS)
                            .agg(F.sum("amount").cast("decimal(18,2)").alias("amount"),
                                 F.sum("n_txn").cast("int").alias("n_txn")))
with timed("weekly: roll-up from daily + write"):
    write_agg(roll("week_start", F.date_trunc("week", "txn_date").cast("date")), P["weekly"], scale=0.8)
with timed("monthly: roll-up from daily + write"):
    write_agg(roll("month_start", F.trunc("txn_date", "month")), P["monthly"], scale=0.5)

with timed("read-back stats"):
    ds = (d.groupBy("category").agg(
            F.count("*").alias("daily_keys"), F.sum("n_txn").alias("rows_after_dedup"),
            F.sum((F.col("src_id").isNull() | F.col("dst_id").isNull()).cast("long")).alias("null_id_keys"),
            F.sum(F.when((F.col("src_kind") == "CUST") & (F.col("dst_kind") == "CUST") & (F.col("src_id") == F.col("dst_id")),
                         F.col("n_txn")).otherwise(0)).alias("self_rows"))
          .toPandas())
    ws = spark.read.parquet(P["weekly"]).groupBy("category").count().withColumnRenamed("count", "weekly_keys").toPandas()
    ms = spark.read.parquet(P["monthly"]).groupBy("category").count().withColumnRenamed("count", "monthly_keys").toPandas()
BYTES = {g: bytes_by_category(p) for g, p in P.items()}

## §4 Per category: how much aggregation buys, and bytes per row

In [ ]:
S = RAW.merge(ds, on="category", how="left").merge(ws, on="category", how="left").merge(ms, on="category", how="left")
for g in P:
    S[f"{g}_bytes"] = S.category.map(lambda c: BYTES[g].get(c, (0, 0))[0])
    S[f"{g}_files"] = S.category.map(lambda c: BYTES[g].get(c, (0, 0))[1])
S = S.fillna({"daily_keys": 0, "weekly_keys": 0, "monthly_keys": 0, "rows_after_dedup": 0})
for g in P:
    S[f"{g}_vs_rows"] = S[f"{g}_keys"] / S.rows_after_dedup.clip(lower=1)
S["daily_B_per_row"] = S.daily_bytes / S.daily_keys.clip(lower=1)

cols = ["category", "card", "rows_after_dedup", "daily_keys", "daily_vs_rows", "weekly_vs_rows", "monthly_vs_rows",
        "daily_B_per_row", "null_id_keys", "self_rows"]
show(S[cols].astype({c: "int64" for c in ["rows_after_dedup", "daily_keys", "null_id_keys", "self_rows"]}),
     "Keys kept per grain as a share of source rows; compressed bytes per daily row",
     pct=("daily_vs_rows", "weekly_vs_rows", "monthly_vs_rows"))

## §5 Storage: probe month and full history, with vs. without card

Full history is extrapolated **per category**: probe bytes × (category's full-history raw bytes ÷ its probe-month raw bytes).
This follows each category's own growth, rather than assuming every month looks like the probe month.

In [ ]:
S["scale"] = S.raw_full_bytes / S.raw_probe_bytes.where(S.raw_probe_bytes > 0)
no_probe = S[S.raw_probe_bytes == 0]
if len(no_probe):
    print("⚠ categories with history but no rows in the probe window (excluded from extrapolation):", list(no_probe.category))

out = []
for lbl, m in (("with card", S.card | ~S.card), ("without card", ~S.card)):
    x = S[m]
    raw_full = x.raw_full_bytes.sum()
    for g in ["daily", "weekly", "monthly", "daily+monthly"]:
        gs = ["daily", "monthly"] if g == "daily+monthly" else [g]
        probe = sum(x[f"{k}_bytes"].sum() for k in gs)
        full  = sum((x[f"{k}_bytes"] * x.scale).sum() for k in gs)
        out.append({"scenario": lbl, "grain": g, "probe_GB": probe / GB, "full_history_GB": full / GB,
                    "hdfs_consumed_GB": full * REPL / GB, "pct_of_remaining": full * REPL / REMAIN_RAW,
                    "vs_raw_table": full / raw_full})
O = pd.DataFrame(out)
show(O, f"Aggregate storage ({CODEC}); consumed = × replication {REPL}; remaining = {REMAIN_RAW/GB:,.0f} GB",
     pct=("pct_of_remaining", "vs_raw_table"), gb=("probe_GB", "full_history_GB", "hdfs_consumed_GB"))
print(f"raw staging table, full history: with card {S.raw_full_bytes.sum()/GB:,.0f} GB | "
      f"without card {S[~S.card].raw_full_bytes.sum()/GB:,.0f} GB (logical)")

## §6 Build time, and extrapolation to full history

Time scales with raw bytes read. The no-card figure assumes time is proportional to bytes, which is approximate: card rows are simple, but they are 44% of rows.

In [ ]:
T = pd.DataFrame(TIMINGS)
show(T.assign(minutes=T.seconds / 60)[["stage", "minutes"]], "Probe stage timings")
build = T[T.stage.str.startswith(("dedup", "daily", "weekly", "monthly"))].seconds.sum()
probe_raw = S.raw_probe_bytes.sum()
ext = pd.DataFrame([
    {"scenario": "with card",    "probe_month_min": build / 60,
     "full_history_hours": build * S.raw_full_bytes.sum() / probe_raw / 3600},
    {"scenario": "without card", "probe_month_min": build * S[~S.card].raw_probe_bytes.sum() / probe_raw / 60,
     "full_history_hours": build * S[~S.card].raw_full_bytes.sum() / probe_raw / 3600},
])
show(ext, "Build time: dedup + daily + weekly + monthly (proportional to raw bytes; cluster load not controlled)")

In [ ]:
if CLEANUP:
    fs_for(PROBE_DIR).delete(JPath(PROBE_DIR), True)
    print("probe output deleted:", PROBE_DIR)
DROP_IDS.unpersist()